<a href="https://colab.research.google.com/github/manuelcernar-rgb/unmsm2026ml/blob/main/Session01_Search_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import heapq
import pandas as pd
from itertools import count

# --- 1. DEFINICIÓN DEL PROBLEMA ---
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def neighbors(state):
    i = state.index(0)
    r, c = divmod(i, 3)
    moves = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            j = nr * 3 + nc
            s = list(state)
            s[i], s[j] = s[j], s[i]
            moves.append(tuple(s))
    return moves

def manhattan(state):
    dist = 0
    for idx, val in enumerate(state):
        if val == 0: continue
        gr, gc = divmod(val - 1, 3)
        r, c = divmod(idx, 3)
        dist += abs(gr - r) + abs(gc - c)
    return dist

def reconstruct(parent, state):
    path = [state]
    while parent[path[-1]] is not None:
        path.append(parent[path[-1]])
    return list(reversed(path))

# --- 2. LOS TRES ALGORITMOS DE BÚSQUEDA ---
def bfs(start):
    frontier, visited = [start], {start}
    parent = {start: None}
    expanded = 0
    while frontier:
        state = frontier.pop(0)
        expanded += 1
        if state == GOAL:
            return reconstruct(parent, state), expanded
        for nxt in neighbors(state):
            if nxt not in visited:
                visited.add(nxt); parent[nxt] = state; frontier.append(nxt)
    return None, expanded

def dfs(start):
    frontier, visited = [start], {start}
    parent = {start: None}
    expanded = 0
    while frontier:
        state = frontier.pop()
        expanded += 1
        if state == GOAL:
            return reconstruct(parent, state), expanded
        for nxt in neighbors(state):
            if nxt not in visited:
                visited.add(nxt); parent[nxt] = state; frontier.append(nxt)
    return None, expanded

def astar(start):
    counter = count()
    frontier = [(manhattan(start), next(counter), start, 0)]
    parent, g_score = {start: None}, {start: 0}
    expanded = 0
    while frontier:
        _, _, state, g = heapq.heappop(frontier)
        expanded += 1
        if state == GOAL:
            return reconstruct(parent, state), expanded
        for nxt in neighbors(state):
            new_g = g + 1
            if nxt not in g_score or new_g < g_score[nxt]:
                g_score[nxt] = new_g
                parent[nxt] = state
                heapq.heappush(frontier, (new_g + manhattan(nxt), next(counter), nxt, new_g))
    return None, expanded

# --- 3. FUNCIONES PARA DIBUJO COMPACTO HORIZONTAL ---
def estado_a_lineas(state):
    """Convierte un estado de 9 números en 3 cadenas de texto (las 3 filas)."""
    lineas = []
    for i in range(0, 9, 3):
        fila = state[i:i+3]
        lineas.append(" ".join(str(x) if x != 0 else "_" for x in fila))
    return lineas

def dibujar_ruta_compacta(path, columnas=5):
    """Imprime la ruta dibujando varios tableros uno al lado del otro."""
    total_pasos = len(path) - 1
    for i in range(0, len(path), columnas):
        chunk = path[i:i+columnas]

        # 1. Crear e imprimir los encabezados
        encabezados = []
        for j in range(len(chunk)):
            paso = i + j
            if paso == 0:
                titulo = f"P{paso} (Ini)"
            elif paso == total_pasos:
                titulo = f"P{paso} (Fin)"
            else:
                titulo = f"Paso {paso}"
            encabezados.append(titulo.ljust(15))
        print("".join(encabezados))

        # 2. Imprimir las 3 filas horizontales de los tableros
        for fila_idx in range(3):
            fila_texto = ""
            for estado in chunk:
                lineas_estado = estado_a_lineas(estado)
                fila_texto += lineas_estado[fila_idx].ljust(15)
            print(fila_texto)

        # 3. Línea en blanco separadora
        print()

# --- 4. EJECUCIÓN MÚLTIPLE Y RECOPILACIÓN DE DATOS ---
estados_prueba = [
    (1, 7, 6, 2, 0, 8, 4, 5, 3),
    (1, 6, 4, 8, 5, 7, 2, 0, 3),
    (5, 3, 1, 4, 0, 8, 2, 6, 7),
    (2, 1, 4, 3, 5, 6, 7, 0, 8),
    (0, 6, 2, 5, 3, 8, 7, 4, 1)
]

# MOSTRAR LA LISTA DE CASOS AL INICIO
print("#"*70)
print("### LISTA DE CASOS A EVALUAR ###")
print("#"*70)
for idx, estado in enumerate(estados_prueba, 1):
    print(f"Caso {idx}: {estado}")
print("#"*70 + "\n")

resultados_globales = []

for idx, start in enumerate(estados_prueba, 1):
    print(f"\n{'='*70}")
    print(f"EVALUANDO CASO DE PRUEBA {idx}: {start}")
    print(f"{'='*70}")

    # Ejecutamos los algoritmos
    path_bfs, exp_bfs = bfs(start)
    path_dfs, exp_dfs = dfs(start)
    path_astar, exp_astar = astar(start)

    # --- RESUMEN INDIVIDUAL DEL CASO ---
    print("\n--- RESUMEN DEL CASO ---")
    print(f"BFS   -> Longitud de ruta: {len(path_bfs)-1}, Nodos expandidos: {exp_bfs}")
    print(f"DFS   -> Longitud de ruta: {len(path_dfs)-1}, Nodos expandidos: {exp_dfs}")
    print(f"A*    -> Longitud de ruta: {len(path_astar)-1}, Nodos expandidos: {exp_astar}")

    # Guardamos los datos para la tabla final
    estado_str = str(start)
    resultados_globales.append({"Caso": idx, "Estado Inicial": estado_str, "Algoritmo": "BFS", "Ruta": len(path_bfs)-1, "Nodos": exp_bfs})
    resultados_globales.append({"Caso": idx, "Estado Inicial": estado_str, "Algoritmo": "DFS", "Ruta": len(path_dfs)-1, "Nodos": exp_dfs})
    resultados_globales.append({"Caso": idx, "Estado Inicial": estado_str, "Algoritmo": "A*", "Ruta": len(path_astar)-1, "Nodos": exp_astar})

    # Dibujamos la solución de forma compacta (5 pasos por línea)
    print("\n--- SOLUCIÓN ÓPTIMA PASO A PASO (A*) ---")
    dibujar_ruta_compacta(path_astar, columnas=5)

# --- 5. TABLA FINAL RESUMEN ORDENADA ---
print("\n" + "#"*70)
print("### TABLA FINAL RESUMEN (ORDENADA POR RUTA Y LUEGO NODOS) ###")
print("#"*70 + "\n")

df_final = pd.DataFrame(resultados_globales)
df_ordenado = df_final.sort_values(by=["Ruta", "Nodos"], ascending=[True, True])
df_ordenado = df_ordenado.reset_index(drop=True)
print(df_ordenado.to_string())

######################################################################
### LISTA DE CASOS A EVALUAR ###
######################################################################
Caso 1: (1, 7, 6, 2, 0, 8, 4, 5, 3)
Caso 2: (1, 6, 4, 8, 5, 7, 2, 0, 3)
Caso 3: (5, 3, 1, 4, 0, 8, 2, 6, 7)
Caso 4: (2, 1, 4, 3, 5, 6, 7, 0, 8)
Caso 5: (0, 6, 2, 5, 3, 8, 7, 4, 1)
######################################################################


EVALUANDO CASO DE PRUEBA 1: (1, 7, 6, 2, 0, 8, 4, 5, 3)

--- RESUMEN DEL CASO ---
BFS   -> Longitud de ruta: 20, Nodos expandidos: 48630
DFS   -> Longitud de ruta: 12514, Nodos expandidos: 12864
A*    -> Longitud de ruta: 20, Nodos expandidos: 635

--- SOLUCIÓN ÓPTIMA PASO A PASO (A*) ---
P0 (Ini)       Paso 1         Paso 2         Paso 3         Paso 4         
1 7 6          1 _ 6          1 6 _          1 6 8          1 6 8          
2 _ 8          2 7 8          2 7 8          2 7 _          2 7 3          
4 5 3          4 5 3          4 5 3          4 5 3    